# QUESTÃO 2-3: Tratamento de Dados e Feature Engineering

## Seção 0: Imports e Configuração

In [1]:
"""
Imports e Configuração do Notebook
==================================
Carrega todas as dependências para tratamento de dados e feature engineering.
"""

import os
import sys
import warnings
from pathlib import Path
import json

warnings.filterwarnings('ignore')

# Configuração de diretórios
ROOT_DIR = Path.cwd()
if ROOT_DIR.name != "Desafio-Lighthouse-Dados-AI":
    ROOT_DIR = ROOT_DIR.parent

SRC_DIR = ROOT_DIR / "src"
DATA_RAW = ROOT_DIR / "data" / "raw"
DATA_PROCESSED = ROOT_DIR / "data" / "processed"

# Criar diretório de saída
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Adiciona a pasta src ao path
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Importações de análise de dados
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Configuração de visualização
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", "{:.2f}".format)

# Importação de funções customizadas
try:
    from data.load_data import load_produtos, load_custos
    from data.clean_data import clean_produtos
    from utils import salvar_csv, garantir_pasta
    print("✓ Módulos carregados com sucesso")
except ImportError as e:
    print(f"✗ Erro ao importar módulos: {e}")
    raise

✓ Módulos carregados com sucesso


---

# QUESTÃO 2 - NORMALIZAÇÃO DE PRODUTOS

## Objetivo
- Carregar `produtos_raw.csv`
- Normalizar e categorizar produtos
- Tratar dados ausentes
- Exportar para `data/processed/produtos_normalizado.csv`

## Carregamento e Limpeza de Produtos

In [2]:
"""
Questão 2: Normalização de Produtos
====================================
Carrega, valida e normaliza o arquivo de produtos.
"""

print("="*60)
print("QUESTÃO 2 - NORMALIZAÇÃO DE PRODUTOS")
print("="*60)

try:
    # Carregar dados brutos
    df_produtos = load_produtos()
    print(f"✓ Produtos carregados: {df_produtos.shape[0]} produtos")
    print(f"✓ Colunas: {', '.join(df_produtos.columns.tolist())}\n")
    
    # Informações iniciais
    print("DADOS BRUTOS:")
    print(f"  Shape: {df_produtos.shape}")
    print(f"  Nulos: {df_produtos.isna().sum().sum()}")
    print(f"\n  Primeiras 5 registros:")
    display(df_produtos.head())
    
except Exception as e:
    print(f"✗ Erro ao carregar produtos: {e}")
    raise

QUESTÃO 2 - NORMALIZAÇÃO DE PRODUTOS
Colunas encontradas em produtos: ['name', 'price', 'code', 'actual_category']
Shape de produtos: (157, 4)
✓ Produtos carregados: 157 produtos
✓ Colunas: name, price, code, actual_category

DADOS BRUTOS:
  Shape: (157, 4)
  Nulos: 0

  Primeiras 5 registros:


,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz


In [3]:
"""
Limpeza e Normalização
=====================
Aplica funções de limpeza e normalização configuradas em src/data/clean_data.py
"""

# Aplicar limpeza
df_produtos_limpo = clean_produtos(df_produtos)

print("\n" + "="*60)
print("DADOS APÓS LIMPEZA E NORMALIZAÇÃO")
print("="*60)
print(f"\n  Shape: {df_produtos_limpo.shape}")
print(f"  Nulos: {df_produtos_limpo.isna().sum().sum()}")
print(f"\n  Tipos de dados:")
print(df_produtos_limpo.dtypes)

# Verificação de categorização
if 'categoria' in df_produtos_limpo.columns:
    print(f"\n  Distribuição de categorias:")
    print(df_produtos_limpo['categoria'].value_counts())

# Estatísticas de preço
if 'preco' in df_produtos_limpo.columns:
    print(f"\n  Estatísticas de Preço:")
    print(df_produtos_limpo['preco'].describe())

print(f"\n  Últimos 5 registros:")
display(df_produtos_limpo.tail())

Duplicatas removidas    : 7
Total de produtos únicos: 150
Categorias finais       : ['ancoragem', 'eletronicos', 'propulsao']

DADOS APÓS LIMPEZA E NORMALIZAÇÃO

  Shape: (150, 4)
  Nulos: 0

  Tipos de dados:
name               object
price             float64
code                int64
actualcategory     object
dtype: object

  Últimos 5 registros:


,name,price,code,actualcategory
152,Corrente Delta Vox Ion,495.98,146,ancoragem
153,Corrente Danforth Force Leviathan Impulse,3030.08,147,ancoragem
154,Âncora Delta Force Barracuda Mako,4785.56,148,ancoragem
155,Cabo de Nylon Bruce Core,1163.62,149,ancoragem
156,Cabo de Nylon Danforth Magnum Vox,1645.66,150,ancoragem


In [6]:
"""
Exportação de Produtos Normalizados
====================================
"""

# Salvar arquivo processado
output_path = DATA_PROCESSED / "produtos_normalizado.csv"
salvar_csv(df_produtos_limpo, output_path)

print(f"\n✓ Arquivo salvo em: {output_path}")
print(f"✓ Questão 2 CONCLUÍDA")

Arquivo salvo em: c:\Projetos\Desafio-Lighthouse-Dados-AI\data\processed\produtos_normalizado.csv

✓ Arquivo salvo em: c:\Projetos\Desafio-Lighthouse-Dados-AI\data\processed\produtos_normalizado.csv
✓ Questão 2 CONCLUÍDA


---

# QUESTÃO 3 - TRANSFORMAÇÃO DE CUSTOS

## Objetivo
- Carregar `custos_importacao.json`
- Transformar estrutura JSON → DataFrame
- Validar e padronizar dados
- Exportar para `data/processed/custos_importacao.csv`

## Carregamento e Transformação de Custos

In [4]:
"""
Questão 3: Transformação de Custos (JSON → CSV)
================================================
Carrega JSON e converte para formato tabular.
"""

print("\n" + "="*60)
print("QUESTÃO 3 - TRANSFORMAÇÃO CUSTOS JSON → CSV")
print("="*60)

try:
    # Carregar dados de custos
    df_custos = load_custos()
    print(f"\n✓ Custos carregados: {df_custos.shape[0]} registros")
    print(f"✓ Colunas: {', '.join(df_custos.columns.tolist())}\n")
    
    # Informações iniciais
    print("DADOS DE CUSTOS:")
    print(f"  Shape: {df_custos.shape}")
    print(f"  Nulos: {df_custos.isna().sum().sum()}")
    print(f"\n  Tipos de dados:")
    print(df_custos.dtypes)
    
    print(f"\n  Primeiras 5 registros:")
    display(df_custos.head())
    
except Exception as e:
    print(f"✗ Erro ao carregar custos: {e}")
    raise


QUESTÃO 3 - TRANSFORMAÇÃO CUSTOS JSON → CSV
Colunas encontradas em custos: ['product_id', 'product_name', 'category', 'historic_data']
Shape de custos: (150, 4)

✓ Custos carregados: 150 registros
✓ Colunas: product_id, product_name, category, historic_data

DADOS DE CUSTOS:
  Shape: (150, 4)
  Nulos: 0

  Tipos de dados:
product_id        int64
product_name     object
category         object
historic_data    object
dtype: object

  Primeiras 5 registros:


,product_id,product_name,category,historic_data
0,1,Transponder AIS Maré Magnum,eletrônicos,"[{'start_date': '10/08/2016', 'usd_price': 105..."
1,2,Transponder Furuno Marlin,eletrônicos,"[{'start_date': '23/11/2017', 'usd_price': 432..."
2,3,Radar Furuno Pulse Leviathan,eletrônicos,"[{'start_date': '12/04/2016', 'usd_price': 254..."
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,"[{'start_date': '04/03/2016', 'usd_price': 909..."
4,5,Piloto Automático Furuno Storm,eletrônicos,"[{'start_date': '10/02/2016', 'usd_price': 600..."


In [5]:
"""
Validação e Limpeza de Custos
==============================
"""

# Remover duplicatas se existirem
duplicatas = df_custos.drop(columns=['historic_data']).duplicated().sum()
print(f"\nDuplicatas encontradas: {duplicatas}")

if duplicatas > 0:
    df_custos = df_custos.drop_duplicates()
    print(f"✓ Duplicatas removidas")

# Validações de dados
print(f"\nValidações:")
for col in df_custos.columns:
    if df_custos[col].dtype in ['float64', 'int64']:
        negativos = (df_custos[col] < 0).sum()
        if negativos > 0:
            print(f"  ⚠️  Coluna '{col}': {negativos} valores negativos")
        else:
            print(f"  ✓ Coluna '{col}': Todos os valores válidos")

# Estatísticas
print(f"\nEstatísticas de Custos:")
print(df_custos.describe())


Duplicatas encontradas: 0

Validações:
  ✓ Coluna 'product_id': Todos os valores válidos

Estatísticas de Custos:
       product_id
count      150.00
mean        75.50
std         43.45
min          1.00
25%         38.25
50%         75.50
75%        112.75
max        150.00


In [6]:
"""
Exportação de Custos em CSV
============================
"""

# Salvar arquivo processado
output_path = DATA_PROCESSED / "custos_importacao.csv"
salvar_csv(df_custos, output_path)

print(f"\n✓ Arquivo salvo em: {output_path}")
print(f"✓ Questão 3 CONCLUÍDA")

Arquivo salvo em: c:\Projetos\Desafio-Lighthouse-Dados-AI\data\processed\custos_importacao.csv

✓ Arquivo salvo em: c:\Projetos\Desafio-Lighthouse-Dados-AI\data\processed\custos_importacao.csv
✓ Questão 3 CONCLUÍDA


---

# SEÇÃO FINAL: FEATURE ENGINEERING

## Objetivo
Criar features derivadas e dimensões de tempo para uso em análises subsequentes.

## Feature Engineering Temporal

In [7]:
"""
Feature Engineering: Adição de Colunas Temporais
=================================================
"""

try:
    from data.load_data import load_vendas
    from features.feature_engineering import adicionar_colunas_tempo, construir_calendario
    
    # Carregar vendas
    df_vendas = load_vendas()
    print(f"Vendas carregadas: {df_vendas.shape[0]} registros\n")
    
    # Adicionar features de tempo
    df_vendas = adicionar_colunas_tempo(df_vendas)
    
    print("✓ Colunas temporais adicionadas:")
    print(f"  {', '.join([col for col in df_vendas.columns if col.startswith(('ano', 'mes', 'dia', 'semana'))])}")
    
    # Visualizar resultado
    print(f"\nAmostra com features temporais:")
    display(df_vendas[['data', 'ano', 'mes', 'dia', 'dia_semana', 'total']].head(10))
    
except Exception as e:
    print(f"⚠️  Aviso em feature engineering: {e}")

Colunas encontradas em vendas: ['id', 'id_client', 'id_product', 'qtd', 'total', 'sale_date']
Coluna de data detectada: 'sale_date'
Coluna de total detectada: 'total'
Shape final de vendas: (9895, 6)
Vendas carregadas: 9895 registros

✓ Colunas temporais adicionadas:
  ano, mes, dia, dia_semana

Amostra com features temporais:


,data,ano,mes,dia,dia_semana,total
0,2023-09-10,2023,9,10,Domingo,3405.00
1,2024-09-15,2024,9,15,Domingo,16873.90
2,2024-08-13,2024,8,13,Terça-feira,9475.30
3,2023-02-03,2023,2,3,Sexta-feira,55893.00
4,2024-02-12,2024,2,12,Segunda-feira,451403.90
5,2023-09-26,2023,9,26,Terça-feira,39056.40
6,2024-02-28,2024,2,28,Quarta-feira,34560.05
7,2023-11-07,2023,11,7,Terça-feira,114932.90
8,2024-08-25,2024,8,25,Domingo,12643.55
9,2023-05-07,2023,5,7,Domingo,23254.00


In [8]:
"""
Construção da Dimensão Calendário
==================================
"""

try:
    # Criar dimensão calendário
    data_min = df_vendas['data'].dropna().min()
    data_max = df_vendas['data'].dropna().max()
    
    print(f"Construindo calendário de {data_min.date()} a {data_max.date()}\n")
    
    df_calendario = construir_calendario(data_min, data_max)
    
    print(f"✓ Dimensão calendário criada: {df_calendario.shape[0]} dias")
    print(f"✓ Colunas: {', '.join(df_calendario.columns.tolist())}")
    print(f"\nAmostra:")
    display(df_calendario.head(10))
    
    # Salvar calendário
    calendario_path = DATA_PROCESSED / "dim_calendario.csv"
    salvar_csv(df_calendario, calendario_path)
    print(f"\n✓ Calendário salvo em: {calendario_path}")
    
except Exception as e:
    print(f"⚠️  Aviso ao criar calendário: {e}")

Construindo calendário de 2023-01-01 a 2024-12-31

✓ Dimensão calendário criada: 731 dias
✓ Colunas: data, ano, mes, dia, dia_semana

Amostra:


,data,ano,mes,dia,dia_semana
0,2023-01-01,2023,1,1,Domingo
1,2023-01-02,2023,1,2,Segunda-feira
2,2023-01-03,2023,1,3,Terça-feira
3,2023-01-04,2023,1,4,Quarta-feira
4,2023-01-05,2023,1,5,Quinta-feira
5,2023-01-06,2023,1,6,Sexta-feira
6,2023-01-07,2023,1,7,Sábado
7,2023-01-08,2023,1,8,Domingo
8,2023-01-09,2023,1,9,Segunda-feira
9,2023-01-10,2023,1,10,Terça-feira


Arquivo salvo em: c:\Projetos\Desafio-Lighthouse-Dados-AI\data\processed\dim_calendario.csv

✓ Calendário salvo em: c:\Projetos\Desafio-Lighthouse-Dados-AI\data\processed\dim_calendario.csv


---

## CONCLUSÃO DO NOTEBOOK 02

### Seções Completadas

| Questão | Descrição | Status |
|---------|-----------|--------|
| **Q2** | Normalização de Produtos | Completo |
| **Q3** | Transformação Custos JSON→CSV | Completo |
| **Features** | Feature Engineering Temporal | Completo |

### Arquivos Gerados

- `data/processed/produtos_normalizado.csv`
- `data/processed/custos_importacao.csv`
- `data/processed/dim_calendario.csv`

### Próximos Passos

- **Notebook 03**: Análise de Insights (Q4, Q5, Q6)
- **Notebook 04**: Modelo de Previsão (Q7)
- **Notebook 05**: Sistema de Recomendação (Q8)